# 1일차 3교시 · 분류 모델 평가 지표

유방암 진단 데이터로 **혼동 행렬, Accuracy, Precision, Recall, F1**을 직접 계산하고, 불균형 데이터에서 정확도가 속이는 경우를 확인합니다.

- 셀을 위에서부터 차례로 실행합니다 (`Shift + Enter`).
- [선택] 표시가 있는 셀은 시간이 남을 때 실행합니다.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (confusion_matrix, accuracy_score, precision_score,
                             recall_score, f1_score, classification_report)

## 1. 데이터 살펴보기

**주의**: 이 데이터는 `0 = malignant(악성)`, `1 = benign(양성 종양)`입니다. 우리가 **찾으려는 대상(양성 클래스, positive)은 악성(0)**입니다.

In [ ]:
data = load_breast_cancer()
X, y = data.data, data.target
print("데이터 크기:", X.shape)
print("클래스 이름:", [str(name) for name in data.target_names])
print("클래스별 건수:", {str(name): int(n) for name, n in zip(data.target_names, np.bincount(y))})

## 2. 학습·테스트 분할과 모델 학습

`stratify=y`는 학습·테스트 세트의 클래스 비율을 원본과 같게 맞춥니다.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)

model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print("테스트 건수:", len(y_test))

## 3. 혼동 행렬

행 = 실제, 열 = 예측. `labels=[0, 1]`로 순서를 고정해 **악성(0)을 첫 행·첫 열**에 둡니다.

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
pd.DataFrame(cm,
             index=["실제: 악성(0)", "실제: 양성종양(1)"],
             columns=["예측: 악성(0)", "예측: 양성종양(1)"])

## 4. 공식으로 직접 계산하기

악성을 양성 클래스(positive)로 보면 TP·FN·FP·TN은 아래와 같습니다.

In [ ]:
TP, FN = cm[0, 0], cm[0, 1]
FP, TN = cm[1, 0], cm[1, 1]
print(f"TP={TP}, FN={FN}, FP={FP}, TN={TN}")

accuracy = (TP + TN) / (TP + TN + FP + FN)
precision = TP / (TP + FP)
recall = TP / (TP + FN)
f1 = 2 * precision * recall / (precision + recall)
print(f"Accuracy  = {accuracy:.3f}")
print(f"Precision = {precision:.3f}")
print(f"Recall    = {recall:.3f}")
print(f"F1        = {f1:.3f}")

## 5. scikit-learn 함수로 계산하기 — `pos_label`에 주의

기본값은 **1을 양성 클래스**로 봅니다. 악성을 기준으로 하려면 `pos_label=0`을 지정해야 합니다.

In [ ]:
print("pos_label 지정 안 함 (양성종양 기준):",
      round(precision_score(y_test, y_pred), 3), round(recall_score(y_test, y_pred), 3))
print("pos_label=0 (악성 기준):         ",
      round(precision_score(y_test, y_pred, pos_label=0), 3),
      round(recall_score(y_test, y_pred, pos_label=0), 3),
      round(f1_score(y_test, y_pred, pos_label=0), 3))

In [ ]:
print(classification_report(y_test, y_pred, target_names=["malignant(악성)", "benign(양성종양)"], digits=3))

**classification_report 읽는 법**

- 클래스마다 precision, recall, f1-score, support(실제 건수)가 나옵니다.
- `macro avg`: 클래스별 지표의 단순 평균 / `weighted avg`: 건수로 가중한 평균

## 6. 정확도의 함정: 불균형 데이터

1,000명 중 환자 60명인 데이터에서 **무조건 "정상"이라고만 하는 모델**(`DummyClassifier`)을 만들어 봅니다.

In [ ]:
from sklearn.dummy import DummyClassifier

y_imb = np.array([1] * 60 + [0] * 940)        # 1 = 환자(양성 클래스), 0 = 정상
X_imb = np.zeros((1000, 1))                    # 특성은 의미 없음

dummy = DummyClassifier(strategy="most_frequent").fit(X_imb, y_imb)
pred = dummy.predict(X_imb)
print("정확도:", accuracy_score(y_imb, pred))
print("재현율:", recall_score(y_imb, pred))
print("정밀도:", precision_score(y_imb, pred, zero_division=0), "(양성 예측이 0건이라 정의되지 않음 → 0으로 처리)")

정확도 0.94인데 환자를 **한 명도** 찾지 못합니다. 양성이 드문 문제에서는 반드시 재현율·정밀도·F1을 함께 봅니다.

## 7. [선택] 임계값 바꾸기

`predict_proba`의 열 순서는 `model.classes_`와 같습니다(`[0, 1]`). 첫 열이 **악성일 확률**입니다.

In [ ]:
proba_malignant = model.predict_proba(X_test)[:, 0]
rows = []
for threshold in [0.9, 0.7, 0.5, 0.3, 0.1]:
    pred_t = np.where(proba_malignant >= threshold, 0, 1)
    rows.append({"임계값": threshold,
                 "악성 예측 건수": int((pred_t == 0).sum()),
                 "정밀도": round(precision_score(y_test, pred_t, pos_label=0), 3),
                 "재현율": round(recall_score(y_test, pred_t, pos_label=0), 3)})
pd.DataFrame(rows)

임계값을 낮출수록 악성으로 판정하는 건수가 늘어 **재현율은 오르고 정밀도는 떨어집니다.**

## 8. [선택] ROC-AUC와 평균 정밀도

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score
y_true_malignant = (y_test == 0).astype(int)   # 악성 = 1로 바꿔서 계산
print("ROC-AUC          :", round(roc_auc_score(y_true_malignant, proba_malignant), 4))
print("Average Precision:", round(average_precision_score(y_true_malignant, proba_malignant), 4))

## 9. 연습 문제

혼동 행렬이 TP=30, FP=20, FN=10, TN=940일 때 정확도, 정밀도, 재현율, F1을 아래 셀에서 계산해 보세요.

In [ ]:
TP, FP, FN, TN = 30, 20, 10, 940
# 여기에 계산 코드를 작성하세요